# Prepare Title Embedding Input

## Assumptions
- The ranked top-post file already exists at `data/processed/top_posts_per_subreddit.parquet`.
- We only recover `name` from raw parquet files and do not rebuild ranking in this notebook.
- Embedding is title-only, so the prepared text comes only from `title`.
- Tokenization is used only to measure `token_len` and flag `truncated_by_tokens`; this notebook does not run an embedding model.

## Plan
1. Load the ranked parquet file and re-validate titles for the embedding step.
2. Recover `name` from raw parquet files by joining on stable lookup columns.
3. Normalize and softly truncate titles, measure token length, and write one parquet output file.


In [ ]:
# Import the minimal set of libraries needed for preparation and parquet I/O.
from __future__ import annotations

import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display
from tokenizers.pre_tokenizers import Whitespace

# Ranked input from the previous step and raw parquet files used to recover `name`.
RANKED_INPUT_PATH = Path("data/processed/top_posts_per_subreddit.parquet")
RAW_INPUT_DIR = Path("data/raw/reddit/pushshift_1")
OUTPUT_PATH = Path("data/processed/title_embedding_input.parquet")

# Title validation and preparation limits.
MIN_TITLE_CHARS = 15
MAX_CHARS = 200
MAX_TOKENS = 64

# Only these columns are needed for the title embedding input.
OUTPUT_COLUMNS = [
    "name",
    "subreddit",
    "title",
    "score",
    "num_comments",
    "rank_in_subreddit",
    "title_for_embedding",
    "token_len",
    "truncated_by_chars",
    "truncated_by_tokens",
]

RANKED_COLUMNS = [
    "subreddit",
    "rank_in_subreddit",
    "author",
    "title",
    "score",
    "num_comments",
    "upvote_ratio",
    "created_utc",
]

RAW_LOOKUP_COLUMNS = [
    "name",
    "subreddit",
    "author",
    "title",
    "score",
    "num_comments",
    "upvote_ratio",
    "created_utc",
]

INVALID_TITLE_VALUES = {"", "none", "null", "nan", "[deleted]"}
WHITESPACE_RE = re.compile(r"\s+")
TOKEN_PRETOKENIZER = Whitespace()
LOOKUP_KEYS = [
    "subreddit",
    "author",
    "title",
    "score",
    "num_comments",
    "upvote_ratio",
    "created_utc",
]

RANKED_INPUT_PATH, OUTPUT_PATH


In [ ]:
# Normalize title text and remove unusual control characters.
def normalize_title(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", text)
    cleaned_chars: list[str] = []

    for char in normalized:
        if char in "\t\n\r":
            cleaned_chars.append(" ")
            continue

        if unicodedata.category(char).startswith("C"):
            continue

        cleaned_chars.append(char)

    return WHITESPACE_RE.sub(" ", "".join(cleaned_chars)).strip()


# Trim long text near a whitespace boundary when possible.
def soft_truncate_text(text: str, max_chars: int) -> tuple[str, bool]:
    if len(text) <= max_chars:
        return text, False

    clipped = text[:max_chars].rstrip()
    last_space = clipped.rfind(" ")

    if last_space <= max_chars // 2:
        return clipped, True

    return clipped[:last_space].rstrip(), True


# Read the ranked parquet file from the previous step and re-validate titles.
def load_ranked_posts() -> pd.DataFrame:
    if not RANKED_INPUT_PATH.exists():
        raise FileNotFoundError(f"Missing ranked input file: {RANKED_INPUT_PATH}")

    ranked = pd.read_parquet(RANKED_INPUT_PATH, columns=RANKED_COLUMNS)
    ranked["title"] = ranked["title"].astype("string").map(normalize_title)
    ranked["title_lower"] = ranked["title"].str.lower()
    ranked["title_len"] = ranked["title"].str.len()

    ranked = ranked[
        ranked["title"].notna()
        & ~ranked["title_lower"].isin(INVALID_TITLE_VALUES)
        & ranked["title_len"].ge(MIN_TITLE_CHARS)
    ].copy()

    return ranked.drop(columns=["title_lower", "title_len"]).reset_index(drop=True)


# Map each subreddit to its raw parquet file so we can recover `name`.
def build_raw_file_map() -> dict[str, Path]:
    file_map: dict[str, Path] = {}
    suffix = "_submissions_cleaned.parquet"

    for raw_path in RAW_INPUT_DIR.glob(f"*{suffix}"):
        subreddit = raw_path.name[: -len(suffix)]
        file_map[subreddit.casefold()] = raw_path

    return file_map


# Join one ranked subreddit subset with its raw file to recover `name`.
def recover_name_for_subreddit(ranked_subset: pd.DataFrame, raw_path: Path) -> pd.DataFrame:
    raw_lookup = pd.read_parquet(raw_path, columns=RAW_LOOKUP_COLUMNS)
    raw_lookup["title"] = raw_lookup["title"].astype("string").map(normalize_title)
    raw_lookup = raw_lookup[LOOKUP_KEYS + ["name"]].drop_duplicates(subset=LOOKUP_KEYS, keep="first")

    return ranked_subset.merge(raw_lookup, on=LOOKUP_KEYS, how="left")


# Recover `name` for every ranked row by reading the matching raw parquet file.
def recover_names(ranked: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    raw_file_map = build_raw_file_map()
    recovered_parts: list[pd.DataFrame] = []
    stats = {
        "ranked_rows_before_name_join": len(ranked),
        "subreddits_without_raw_file": 0,
        "rows_without_name_after_join": 0,
    }

    for index, (subreddit, ranked_subset) in enumerate(ranked.groupby("subreddit", sort=False), start=1):
        raw_path = raw_file_map.get(subreddit.casefold())
        if raw_path is None:
            stats["subreddits_without_raw_file"] += 1
            continue

        recovered = recover_name_for_subreddit(ranked_subset.copy(), raw_path)
        recovered_parts.append(recovered)

        if index % 500 == 0:
            print(f"Recovered names for {index} subreddits.")

    if not recovered_parts:
        raise ValueError("Could not recover any rows with `name`.")

    recovered = pd.concat(recovered_parts, ignore_index=True)
    stats["rows_without_name_after_join"] = int(recovered["name"].isna().sum())
    recovered = recovered.dropna(subset=["name"]).copy()

    return recovered.reset_index(drop=True), stats


# Prepare the final title text and measure token length.
def add_embedding_columns(df: pd.DataFrame) -> pd.DataFrame:
    prepared_titles: list[str] = []
    truncated_by_chars: list[bool] = []

    for title in df["title"].tolist():
        prepared_title, was_truncated = soft_truncate_text(title, MAX_CHARS)
        prepared_titles.append(prepared_title)
        truncated_by_chars.append(was_truncated)

    pre_tokenize = TOKEN_PRETOKENIZER.pre_tokenize_str
    token_lengths = [len(pre_tokenize(title)) for title in prepared_titles]

    prepared = df.copy()
    prepared["title_for_embedding"] = prepared_titles
    prepared["token_len"] = token_lengths
    prepared["truncated_by_chars"] = truncated_by_chars
    prepared["truncated_by_tokens"] = prepared["token_len"] > MAX_TOKENS

    return prepared


In [ ]:
# Run the full preparation flow and write one parquet file for the embedding step.
start_time = time.perf_counter()

ranked = load_ranked_posts()
recovered, recovery_stats = recover_names(ranked)
prepared = add_embedding_columns(recovered)
prepared = prepared[OUTPUT_COLUMNS].reset_index(drop=True)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
prepared.to_parquet(OUTPUT_PATH, index=False)

elapsed_seconds = time.perf_counter() - start_time
summary = pd.DataFrame(
    [
        {
            "input_rows": len(ranked),
            **recovery_stats,
            "output_rows": len(prepared),
            "subreddits_in_output": prepared["subreddit"].nunique(),
            "max_rank": int(prepared["rank_in_subreddit"].max()),
            "titles_truncated_by_chars": int(prepared["truncated_by_chars"].sum()),
            "titles_truncated_by_tokens": int(prepared["truncated_by_tokens"].sum()),
            "output_path": str(OUTPUT_PATH),
            "elapsed_seconds": round(elapsed_seconds, 2),
        }
    ]
)

display(summary)
prepared.head(10)


In [ ]:
# Verify the output schema and preview token-related flags.
assert prepared["title_for_embedding"].notna().all()
assert prepared["name"].notna().all()
assert prepared["rank_in_subreddit"].ge(1).all()

preview = prepared[
    [
        "name",
        "subreddit",
        "title",
        "title_for_embedding",
        "token_len",
        "truncated_by_chars",
        "truncated_by_tokens",
    ]
].head(10)

display(preview)
prepared.dtypes.astype(str)
